# الدرس 11 - بروتوكول سياق النموذج (MCP)

يُعَدّ **بروتوكول سياق النموذج (MCP)** معيارًا مفتوحًا يتيح للوكلاء اكتشاف واستخدام الأدوات والموارد ومصادر البيانات ديناميكيًا أثناء وقت التشغيل. بدلاً من ترميز الأدوات داخل الوكيل، يسمح MCP للوكلاء بالاتصال بخوادم خارجية تكشف عن القدرات عند الطلب.

في هذا الدرس، ستتعلم:
- ما هو MCP ولماذا يهم لأنظمة الوكلاء
- كيف تعمل بنية العميل-الخادم في MCP
- كيفية بناء وكلاء يستخدمون اكتشاف الأدوات على نمط MCP


## الإعداد

**المتطلبات الأساسية:**
- مشروع Azure AI Foundry مع نموذج منشور
- قم بتشغيل `az login` للمصادقة


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ما هو بروتوكول سياق النموذج (MCP)؟

يحدد MCP طريقة معيارية لوكلاء الذكاء الاصطناعي لاكتشاف والتفاعل مع الأدوات والمصادر الخارجية للبيانات:

- **MCP Server**: يعرض الأدوات والموارد والمطالبات عبر بروتوكول معياري
- **MCP Client**: بيئة تشغيل الوكيل التي تتصل بالخوادم وتكتشف القدرات المتاحة
- **Dynamic Discovery**: الوكلاء لا يحتاجون إلى أدوات مضمّنة — إنهم يكتشفون ما هو متاح أثناء وقت التشغيل

هذا أمر قوي لبناء أنظمة وكلاء قابلة للتوسيع حيث يمكن إضافة قدرات جديدة دون تعديل كود الوكيل.


## كيف يعمل MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. يتصل الوكيل (عميل MCP) بخادم MCP
2. يستجيب الخادم بقائمة بالأدوات المتاحة ومخططاتها
3. ثم يمكن للوكيل استدعاء أي أداة مكتشفة أثناء استدلاله
4. تتدفق النتائج عبر نفس البروتوكول


## محاكاة اكتشاف أدوات MCP

نظرًا لأن خادم MCP الحقيقي يتطلب عملية خادم قيد التشغيل، سنوضح النمط باستخدام دوال `@tool` التي تحاكي ما قد توفّره خدمة الإقامة المتصلة بـ MCP.

في بيئة الإنتاج، سيتم اكتشاف هذه الأدوات ديناميكيًا من خادم MCP بدلًا من تعريفها محليًا.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## بناء وكيل بأدوات على طراز MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP في بيئة الإنتاج

في بيئة إنتاجية، يتيح MCP أنماطًا قوية:

- **Dynamic tool discovery**: يتصل الوكلاء بخوادم MCP ويكتشفون الأدوات في وقت التشغيل
- **Decoupled architecture**: يمكن لمزودي الأدوات التحديث بشكل مستقل عن الوكيل
- **Cross-organization sharing**: يمكن للفرق إتاحة الإمكانيات عبر خوادم MCP بحيث يمكن لأي وكيل استخدامها
- **Microsoft Agent Framework support**: يحتوي MAF على دعم عميل MCP مدمج عبر التكامل `mcp`

To use a real MCP server with MAF, you would connect via `hosted_mcp_tool()` or the MCP client integration.

**تعرف على المزيد:**
- [مواصفة MCP](https://modelcontextprotocol.io/)
- [دعم MCP في Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## الملخص

في هذا الدرس، تعلمت:
- **MCP** هو معيار مفتوح لاكتشاف الأدوات الديناميكي بين الوكلاء ومزودي الأدوات
- تسمح **معمارية العميل-الخادم** للوكلاء باكتشاف القدرات في وقت التشغيل
- **MCP** يمكّن **أنظمة وكلاء قابلة للتوسعة ومفصولة** حيث يمكن إضافة الأدوات دون تغييرات في الكود
- يوفر Microsoft Agent Framework **دعم MCP المدمج** للاستخدام في بيئات الإنتاج


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
إخلاء المسؤولية:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية [Co-op Translator](https://github.com/Azure/co-op-translator). وعلى الرغم من سعينا لتحقيق الدقة، يُرجى ملاحظة أن الترجمات الآلية قد تحتوي على أخطاء أو معلومات غير دقيقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر المرجعي والموثوق. للمعلومات الحساسة أو الحرجة، يُنصح بالاستعانة بترجمة بشرية مهنية. نحن غير مسؤولين عن أي سوء فهم أو تفسيرات خاطئة ناتجة عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
